# 02 — Exploratory Data Analysis

Summary statistics and 19 publication-style figures over the master dataset.
All figures are written to `outputs/figures/eda/`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', context='notebook')

from src.utils.config import FINAL_DIR, VAR_GROUPS, PRIMARY_OUTCOME, PRIMARY_TREATMENT
df = pd.read_csv(FINAL_DIR / 'master_dataset.csv')
print('Shape:', df.shape, '| Countries:', df['iso3'].nunique(), '| Years:', df['year'].min(), '-', df['year'].max())

## Summary statistics by income group

In [ ]:
headline = ['life_expectancy', 'gdp_per_capita_ppp', 'health_exp_pct_gdp',
            'infant_mortality', 'fertility_rate', 'urban_pop_pct']
df.groupby('income_group')[headline].agg(['mean', 'std']).round(2)

## Missingness (post-imputation)

In [ ]:
non_id = [c for c in df.columns if c not in {'iso3','country','year','income_group'}]
miss = (df[non_id].isna().mean() * 100).sort_values(ascending=False)
print('Total cells:', df[non_id].size, '| % missing:', round(df[non_id].isna().mean().mean()*100, 2))
miss.head(15).round(2)

## Generate all 19 EDA figures
Each function returns the saved file path so figures can be inline-displayed.

In [ ]:
from src.visualization import eda
paths = eda.run_all()
print(f'Wrote {len(paths)} figures.')

## Display the figures inline

In [ ]:
from IPython.display import Image, display
for p in sorted(paths):
    display(Image(filename=str(p)))

## Headline correlations with life expectancy

In [ ]:
feat = [c for c in df.columns if c not in {'iso3','country','year','income_group', PRIMARY_OUTCOME}
        and not any(s in c for s in ('_lag', '__x__'))]
corr = df[feat].apply(lambda c: c.corr(df[PRIMARY_OUTCOME]) if c.dtype.kind in 'fi' else np.nan).dropna().sort_values()
print('Top positive:'); print(corr.tail(10).round(3))
print('\nTop negative:'); print(corr.head(10).round(3))